# d3blobgen over low level interface

## d3function decorator

Decorator `@d3function` converts normal callable function into `D3Function` object, which has extra interfaces that can be used to control the plugin pipeline.

The decorator can be put with or without `module_name` argument. Although giving `module_name` is advised in most of cases as it offers much more features, it is important to note that the `module_name` must be registered to Designer side before you can start using it.

On the other hand, `d3function` without `module_name` doesn't require to be registered. It can be sent to Designer plugin endpoint anytime; therefore, it's suitable for simple usecase with a stnadalne function.

### blob and json

The core interface of `D3Function` is `blob` and `json`.

The `json` provide a `json` object you can send to Designer plugin endpoint to call the function on Designer side. This is the bare object you can use with any network interface library such as `requests` or `aiohttp`.

On the other hand `blob` returns `TypedBlob`, which is superset of `json` object. It offers not only `json`, but also `return_type` and `module_name`. Combining `blob` with `d3_api_plugin` or `d3_api_aplugin` will help you to leverage rich typing and Exception with good details.





## Get blob for requests

- `blob` is an object that provides:
    - `json`: data to send to plugin endpoint
    - `return_type`: type information of `returnValue` expected from Designer plugin endpoint
    - `module_name`: module name of the d3function if it has one

#### d3function example without module

In [ ]:

from d3blobgen.core import d3function

@d3function
def my_add(a: int, b: int) -> int:
    return a + b

In [ ]:
from d3blobgen.core import TypedPayload

blob: TypedPayload[int] = my_add.payload(1, 2)
print(blob)

TypedBlob(json={'script': 'a=1\nb=2\nreturn a + b\n'}, return_type=<class 'int'>, module_name='')


In [7]:
print(blob.script)

aa


#### d3function example with module

In [5]:
from d3blobgen.core import d3function

@d3function(module_name="mymodule")
def my_add(a: int, b: int) -> int:
    return a + b

In [ ]:
from d3blobgen.core import TypedPayload

blob: TypedPayload[int] = my_add.payload(1, 2)
print(blob)

TypedBlob(json={'moduleName': 'mymodule', 'script': 'return my_add(1, 2)'}, return_type=<class 'int'>, module_name='mymodule')


In [7]:
print(blob.json["script"])

return my_add(1, 2)


## Get plugin URL for requests

To call your python function on Designer side over plugin system, the json data must be sent to specific endpoint.
The endpoint can be retrived by d3blobgen api:
- `get_plugin_module_register_url`
- `get_plugin_endpoint_url`

If defined `d3function` has a `module_name`, it must register `d3function` with `module_name` first.
Otherwise, Designer will fail to run the python over plugin system.

#### Registering d3function module

In [14]:
import requests

from d3blobgen.core import d3function, D3Function
from d3blobgen.api import get_plugin_module_register_url

@d3function(module_name="mymodule")
def my_add(a: int, b: int) -> int:
    return a + b

DESIGNER_IP = "localhost"
DESIGNER_PORT = 80
register_url: str = get_plugin_module_register_url(DESIGNER_IP, DESIGNER_PORT)
json: dict[str, str] | None = D3Function.get_module_register_json("mymodule")
print("json:")
print(json)

response = requests.post(register_url, json=json)
print("repond:")
print(response.text)

json:
{'moduleName': 'mymodule', 'contents': '\n\ndef my_add(a, b):\n    return a + b'}
repond:
{"status":{"code":0,"message":"","details":[]}}


#### Post d3function execution

With raw json (no type information)

In [24]:
from d3blobgen.api import get_plugin_endpoint_url, d3_api_plugin_raw
from d3blobgen.models import PluginResponse

DESIGNER_IP = "localhost"
DESIGNER_PORT = 80
json = my_add.json(1, 2)
print("json:")
print(json)

# Get respond from d3_api_plugin_raw
response: PluginResponse = d3_api_plugin_raw(DESIGNER_IP, DESIGNER_PORT, json)
print("reponse:")
print(response)
print("returnValue:")
print(response.returnValue)

print("returnValue with type check:")
returnValue: int = response.returnCastValue(int)
print(type(returnValue))
print(returnValue)

# Get respond from requests
plugin_url: str = get_plugin_endpoint_url(DESIGNER_IP, DESIGNER_PORT)
print("plugin_url:")
print(plugin_url)
requests_response = requests.post(plugin_url, json=json)
print("reponse:")
print(requests_response.text)
print("returnValue:")
print(requests_response.json().get("returnValue"))

json:
{'moduleName': 'mymodule', 'script': 'return my_add(1, 2)'}
reponse:
status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 5.0449ms\n' pythonLog='' returnValue=3
returnValue:
3
returnValue with type check:
<class 'int'>
3
plugin_url:
http://localhost:80/api/session/python/execute
reponse:
{"status":{"code":0,"message":"","details":[]},"d3Log":"Python script took 4.6091ms\n","pythonLog":"","returnValue":"3"}
returnValue:
3


With blob (type information)

In [ ]:
from d3blobgen.api import d3_api_plugin
from d3blobgen.models import PluginResponse, TypedPayload

DESIGNER_IP = "localhost"
DESIGNER_PORT = 80
plugin_url: str = get_plugin_endpoint_url(DESIGNER_IP, DESIGNER_PORT)
blob: TypedPayload[int] = my_add.payload(1, 2)
print("blob:")
print(blob)

response: PluginResponse[int] = d3_api_plugin(DESIGNER_IP, DESIGNER_PORT, blob)
print("repond:")
print(response)

print("returnValue:")
print(response.returnValue)

blob:
TypedBlob(json={'moduleName': 'mymodule', 'script': 'return my_add(1, 2)'}, return_type=<class 'int'>, module_name='mymodule')
repond:
status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 4.8578ms\n' pythonLog='' returnValue=3
returnValue:
3


#### Exception handling

`d3_api_plugin` will raise `PluginException` if plugin fails on Designer side:

In [ ]:
from d3blobgen.api import d3_api_plugin
from d3blobgen.models import PluginResponse, PluginException

@d3function
def my_exception_handling():
    raise RuntimeError("This is my exeption!")

DESIGNER_IP = "localhost"
DESIGNER_PORT = 80
blob: TypedPayload = my_exception_handling.payload()
print("blob:")
print(blob)

try:
    response: PluginResponse[int] = d3_api_plugin(DESIGNER_IP, DESIGNER_PORT, blob)
    print("repond:")
    print(response)
except PluginException as e:
    print(e)

blob:
TypedBlob(json={'script': "\nraise RuntimeError('This is my exeption!')\n"}, return_type=typing.Any, module_name='')

D3PluginError:
- code       : 5000
- messages   :
Failed to run plugin: default_plugin
Error code: 0, Error message: class PythonException in 'C:\dev\d3\d3python\pythonobject.cpp' (line 32):
	
Traceback (most recent call last):

  File "<string>", line 13, in <module>

  File "<string>", line 11, in userScript

RuntimeError: This is my exeption!
.

- d3Log      : None
- pythonLog  : None
- Traceback  :
File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\dev\plugin\Designer_Plugin-ColourCal\backend\external\d3blobgen\examples\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\dev\plugin\Designer_Plugin-ColourCal\backend\external\d3blobgen\examples\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.star

## Async examples

d3blobgen supports async implementaion as well!

In [ ]:
from d3blobgen.core import d3function, D3Function
from d3blobgen.api import d3_api_aregister_module, d3_api_aplugin_raw, d3_api_aplugin
from d3blobgen.models import PluginResponse, TypedPayload

@d3function(module_name="mymodule")
def my_add(a: int, b: int) -> int:
    return a + b

DESIGNER_IP = "localhost"
DESIGNER_PORT = 80
register_url: str = get_plugin_module_register_url(DESIGNER_IP, DESIGNER_PORT)
json: dict[str, str] | None = D3Function.get_module_register_json("mymodule")

print("=" * 60)
print("register:")
print("=" * 60)
print("json:")
print(json)

response = await d3_api_aregister_module(DESIGNER_IP, DESIGNER_PORT, json)
print("response:")
print(response)

print("=" * 60)
print("execute:")
print("=" * 60)

# json implementation without typing information
plugin_json: dict[str, str] = my_add.json(1, 2)
plugin_response: PluginResponse = await d3_api_aplugin_raw(DESIGNER_IP, DESIGNER_PORT, plugin_json)
print("response:")
print(plugin_response)
print("returnValue:")
print(plugin_response.returnValue)

# blob implementation with typing information
blob: TypedPayload[int] = my_add.payload(1, 2)
blob_response: PluginResponse[int] = await d3_api_aplugin(DESIGNER_IP, DESIGNER_PORT, blob)
print("response:")
print(blob_response)
print("returnValue:")
returnValue: int = blob_response.returnValue
print(returnValue)

register:
json:
{'moduleName': 'mymodule', 'contents': '\n\ndef my_add(a, b):\n    return a + b'}
response:
status=PluginStatus(code=0, message='', details=[])
execute:
response:
status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 5.0655ms\n' pythonLog='' returnValue=3
returnValue:
3
response:
status=PluginStatus(code=0, message='', details=[]) d3Log='Python script took 4.938599999999999ms\n' pythonLog='' returnValue=3
returnValue:
3


#### Exception handling

In [ ]:
@d3function
def my_exception_handling():
    raise RuntimeError("This is my exeption!")

DESIGNER_IP = "localhost"
DESIGNER_PORT = 80
blob: TypedPayload = my_exception_handling.payload()
print("blob:")
print(blob)

try:
    response: PluginResponse[int] = await d3_api_aplugin(DESIGNER_IP, DESIGNER_PORT, blob)
    print("repond:")
    print(response)
except PluginException as e:
    print(e)

blob:
TypedBlob(json={'script': "\nraise RuntimeError('This is my exeption!')\n"}, return_type=typing.Any, module_name='')

D3PluginError:
- code       : 5000
- messages   :
Failed to run plugin: default_plugin
Error code: 0, Error message: class PythonException in 'C:\dev\d3\d3python\pythonobject.cpp' (line 32):
	
Traceback (most recent call last):

  File "<string>", line 13, in <module>

  File "<string>", line 11, in userScript

RuntimeError: This is my exeption!
.

- d3Log      : None
- pythonLog  : None
- Traceback  :
File "<frozen runpy>", line 198, in _run_module_as_main
  File "<frozen runpy>", line 88, in _run_code
  File "c:\dev\plugin\Designer_Plugin-ColourCal\backend\external\d3blobgen\examples\.venv\Lib\site-packages\ipykernel_launcher.py", line 18, in <module>
    app.launch_new_instance()
  File "c:\dev\plugin\Designer_Plugin-ColourCal\backend\external\d3blobgen\examples\.venv\Lib\site-packages\traitlets\config\application.py", line 1075, in launch_instance
    app.star

In [1]:
from d3blobgen.client import D3PluginClient

class MyClient(D3PluginClient):
    def __init__(self):
        pass

    async def my_async_fn(self, a: int, b: str) -> str:
        async def inner_fn(c: int) -> str:
            return f"{c}"
        return await inner_fn(a)
    
    async def my_await_fn(self, a: int, b: str) -> str:
        return await self.my_async_fn(a, b)
    
client = MyClient()
print(client._get_register_module_content())

class MyClient:

    def __init__(self):
        pass

    def my_async_fn(self, a, b):

        def inner_fn(c):
            return f'{c}'
        return inner_fn(a)

    def my_await_fn(self, a, b):
        return self.my_async_fn(a, b)

plugin = MyClient()


In [ ]:
from d3blobgen.core import d3function

@d3function("module")
async def my_fn(
    regular: str,           # NOT removed by manual loop (handled by visit_arg)
    val: int,
    *args: int,             # ← Line 257-258 removes this annotation
    option: bool = True,    # ← Line 253-254 removes this annotation (kwonly)
    **kwargs: dict          # ← Line 261-262 removes this annotation
) -> str:                   # Removed elsewhere (line 229)
    def __init__(self):
        pass
    async def inner(x: int, y: str) -> str:  # ← visit_arg removes x: int and y: str
        async def inner2(a: int, b: int) -> str:
            return f"{a}, {b}"
        return await inner2(1, 2)
    return await inner(1, '2')

print(my_fn.function_info.blob_py27)

def my_fn(regular, val, *args, option=True, **kwargs):

    def __init__(self):
        pass

    def inner(x, y):

        def inner2(a, b):
            return f'{a}, {b}'
        return inner2(1, 2)
    return inner(1, '2')


In [1]:
import inspect
import ast
import d3blobgen.ast_utils as ast_utils
from d3blobgen.ast_utils import ConvertToPython27

async def asyncfn() -> str:
    a: int = 2
    async def innerfn():
        return "hello"
    return await innerfn()

# Remove common leading whitespace to handle functions defined with indentation
tree = ast.parse(inspect.getsource(asyncfn))

first_node = tree.body[0]
print(ast.unparse(first_node))

# transformer = ConvertToPython27()
# new_func = transformer.visit(func_node)
first_node = ast_utils.convert_function_to_py27(first_node)
print(ast.unparse(first_node))

async def asyncfn() -> str:
    a: int = 2

    async def innerfn():
        return 'hello'
    return await innerfn()
def asyncfn():
    a = 2

    def innerfn():
        return 'hello'
    return innerfn()


In [2]:
import inspect
import ast
from d3blobgen.ast_utils import ConvertToPython27, convert_function_to_py27
async def asyncfn() -> str:
    a: int = 2
    async def innerfn():
        return "hello"
    return await innerfn()

# Remove common leading whitespace to handle functions defined with indentation
tree = ast.parse(inspect.getsource(asyncfn))

first_node = tree.body[0]
first_node = convert_function_to_py27(first_node)
print(ast.unparse(first_node))

transformer = ConvertToPython27()
new_func = transformer.visit(first_node)
print(ast.unparse(new_func))

def asyncfn():
    a = 2

    def innerfn():
        return 'hello'
    return innerfn()
def asyncfn():
    a = 2

    def innerfn():
        return 'hello'
    return innerfn()


In [2]:
from d3blobgen.client import D3PluginClient

class MyClient(D3PluginClient):
    def __init__(self):
        pass

    async def my_async_fn(self, a: int, b: str) -> str:
        async def inner_fn(c: int) -> str:
            return f"{c}"
        return await inner_fn(a)
    
    async def my_await_fn(self, a: int, b: str) -> str:
        return await self.my_async_fn(a, b)
    
client = MyClient()
print(client._get_register_module_content())

print(client.my_async_fn_payload(1, '2'))

class MyClient:

    def __init__(self):
        pass

    def my_async_fn(self, a, b):

        def inner_fn(c):
            return f'{c}'
        return inner_fn(a)

    def my_await_fn(self, a, b):
        return self.my_async_fn(a, b)

plugin = MyClient()
TypedBlob(json={'moduleName': 'MyClient', 'script': "return plugin.my_async_fn(1, '2')"}, module_name='MyClient')


In [ ]:
from pydantic import BaseModel, Field, field_serializer
from typing import Generic, TypeVar

RetType = TypeVar("RetType")

class Custom(BaseModel, Generic[RetType]):
    # moduleName: str | None = Field(default = None, description="Module name to run script on Deisgner")
    moduleName: str | None = Field(
        default = None,
        exclude_if=lambda v: v is None, 
    )
    script: str = Field(description="Script to run on Designer")

    @field_serializer('moduleName', when_used='unless-none')
    def serialize_module_name(self, value):
        return value


# class TypedBlob(Custom, Generic[RetType]):
#     """Type-safe execution blob for plugin calls.

#     This dataclass packages together the execution blob, expected return type,
#     and module name for type-safe plugin execution.

#     Attributes:
#         blob: The execution blob dictionary (script, moduleName, etc.)
#         module_name: The name of the module this execution belongs to
#     """


mypayload = Custom(moduleName="abc", script="Hello")
print(mypayload.model_dump_json(indent=2))

mypayload2 = Custom(script="Hello")
print(mypayload2.model_dump())

# blob = TypedBlob[int](moduleName="modeul", script="aa")
# print(blob.model_dump())

{
  "moduleName": "abc",
  "script": "Hello"
}
{'script': 'Hello'}
{'moduleName': 'modeul', 'script': 'aa'}
